# Losses, metricas e mascaras

Este notebook explica como o RioNowcast treina com observacoes pontuais de estacoes. Os exemplos sao sinteticos e usam as mesmas losses do pipeline.

## Ideia central

O target contem `log1p(mm/15 min)` nos pixels e instantes observados. A mascara vale um nesses locais e zero no restante da grade. Portanto, ausencia de estacao nao e interpretada como chuva zero e nao entra na loss nem nas metricas.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Nao foi possivel localizar a raiz do repositorio.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from nowcasting.losses import (
    MaskedHuberLoss, MaskedMAELoss, WeightedMaskedHuberLoss, WeightedMaskedMAELoss,
)


## Target em escala logaritmica

A transformacao `log1p` reduz a assimetria da chuva. O modelo e treinado nessa escala; para as metricas operacionais, a previsao e o target retornam a `mm/15 min` com `expm1`.

In [ ]:
rain_mm15 = torch.tensor([0.0, 1.0, 3.0, 8.0, 20.0])
prediction_mm15 = torch.tensor([0.0, 1.5, 2.0, 4.0, 10.0])

target = torch.log1p(rain_mm15).reshape(1, 1, 1, 1, -1)
prediction = torch.log1p(prediction_mm15).reshape(1, 1, 1, 1, -1)

# O terceiro ponto nao foi observado: seu valor nao deve afetar os resultados.
mask = torch.tensor([1.0, 1.0, 0.0, 1.0, 1.0]).reshape(1, 1, 1, 1, -1)

display(pd.DataFrame({
    'chuva_mm15': rain_mm15.numpy(),
    'target_log1p': target.flatten().numpy(),
    'previsao_mm15': prediction_mm15.numpy(),
    'previsao_log1p': prediction.flatten().numpy(),
    'mascara': mask.flatten().numpy(),
}))

## Losses usadas no treinamento

A MAE penaliza linearmente o erro. A Huber e quadratica perto de zero e linear para erros grandes, controlando a influencia de outliers. As versoes ponderadas aumentam a contribuicao de chuva moderada, forte e extrema.

In [ ]:
losses = {
    'masked_mae': MaskedMAELoss(),
    'masked_huber': MaskedHuberLoss(delta=0.1),
    'weighted_mae': WeightedMaskedMAELoss((1, 5, 10, 20)),
    'weighted_huber': WeightedMaskedHuberLoss(delta=0.1, class_weights=(1, 5, 10, 20)),
}

loss_table = pd.DataFrame({
    name: [float(loss(prediction, target, mask))]
    for name, loss in losses.items()
})
display(loss_table.T.rename(columns={0: 'valor_na_escala_log1p'}))

# Alterar um ponto mascarado nao muda nenhuma loss.
prediction_changed = prediction.clone()
prediction_changed[..., 2] = 99.0
assert all(torch.isclose(loss(prediction, target, mask), loss(prediction_changed, target, mask)) for loss in losses.values())
print('Verificado: posicoes mascaradas nao alteram as losses.')

## Pesos por intensidade

Os limiares sao `1,25`, `6,25` e `12,5 mm/15 min`. Os pesos `1, 5, 10, 20` sao aplicados ao target observado. Eles nao criam eventos extremos: apenas aumentam sua contribuicao quando aparecem em um batch.

In [ ]:
thresholds = [1.25, 6.25, 12.5]
weights = [1, 5, 10, 20]
classes = pd.cut(
    rain_mm15.numpy(),
    bins=[-np.inf, *thresholds, np.inf],
    labels=['fraca', 'moderada', 'forte', 'extrema'],
    right=False,
)
weight_by_class = dict(zip(['fraca', 'moderada', 'forte', 'extrema'], weights))
display(pd.DataFrame({
    'chuva_mm15': rain_mm15.numpy(),
    'classe': classes,
    'peso': [weight_by_class[value] for value in classes],
    'observada': mask.flatten().numpy().astype(bool),
}))

## Metricas em unidade fisica

Para interpretar o produto, RMSE, MAE e vies sao calculados depois de `expm1`, somente onde `M=1`. A loss de treinamento e uma funcao de otimizacao; nao deve ser confundida com uma metrica operacional.

In [ ]:
observed = mask.bool()
target_physical = torch.expm1(target)[observed]
prediction_physical = torch.clamp(torch.expm1(prediction), min=0.0)[observed]
error = prediction_physical - target_physical

metrics = {
    'n': int(observed.sum()),
    'rmse_mm15': float(torch.sqrt(torch.mean(error.square()))),
    'mae_mm15': float(torch.mean(error.abs())),
    'bias_mm15': float(torch.mean(error)),
}
display(pd.Series(metrics, name='valor').to_frame())

plt.figure(figsize=(6, 3))
plt.plot(target_physical.numpy(), marker='o', label='observado')
plt.plot(prediction_physical.numpy(), marker='o', label='previsto')
plt.xlabel('Observacao valida')
plt.ylabel('mm/15 min')
plt.title('Metricas calculadas apos inverter log1p')
plt.legend();
plt.tight_layout()

## Cuidados metodologicos

- Nunca substitua ausencias por zero sem manter uma mascara separada.
- Calcule pesos com base no target, nao na previsao.
- Compare modelos pelas mesmas observacoes de teste (`n`) e pelos mesmos horizontes.
- Reporte metricas por intensidade: uma MAE global baixa pode ocultar falha em chuva extrema.
- Mantenha limiares e pesos versionados na configuracao do experimento.